In [2]:
!date

Tue Aug 13 15:49:26 PDT 2024


In [3]:
%load_ext autoreload
%load_ext line_profiler

In [4]:
import logging
logging.basicConfig(level=logging.INFO, force=True)

In [5]:
import os as _os
_os.chdir(_os.environ['PROJECT_ROOT'])

In [6]:
import graph_tool as gt
import graph_tool.draw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from contextlib import contextmanager
import xarray as xr
from itertools import product
from tqdm import tqdm
from itertools import chain
from graph_tool.util import find_edge
import scipy as sp
import fastcluster
from collections import defaultdict

import strainzip as sz
from strainzip.pandas_util import idxwhere
import strainzip.app.unzip

from multiprocessing import Pool

from scipy.cluster.hierarchy import fcluster, linkage
import lib.plot

TODO:
- Read in each unitig
- kmerize
- Make a map from kmers to unitigs
- Read in the merged kmer table
- Build up a map from unitigs to kmer tables
- For each kmer table, estimate the depth across samples
- Write the output

In [54]:
d1 = xr.load_dataarray('data/group/btheta_2strains/r.proc.ggcat-k111-withmegahit2-droptips.unitig_depth.nc~1')
d1

<xarray.DataArray (unitig: 26923, sample: 2)> Size: 431kB
array([[  0.        ,  72.30769231],
       [371.        , 161.66666667],
       [ 36.54901961,  37.09803922],
       ...,
       [ 64.57916325,  62.35890074],
       [ 67.83037722,  62.28728454],
       [ 69.79480346,  70.77267007]])
Coordinates:
  * unitig   (unitig) int64 215kB 0 1 2 3 4 5 ... 26918 26919 26920 26921 26922
  * sample   (sample) <U53 424B 'Bacteroides-thetaiotaomicron-1-1-6_MAF-2_wgs...

In [48]:
sample_list = "xjin_AB_P0R1a,xjin_AB_P0R1b,xjin_AB_P0R1c,xjin_AB_P0R2a,xjin_AB_P0R2b,xjin_AB_P0R2c,xjin_AB_P0R3a,xjin_AB_P0R3b,xjin_AB_P0R3c,xjin_AB_P1R1a,xjin_AB_P1R1b,xjin_AB_P1R1c,xjin_AB_P1R2a,xjin_AB_P1R2b,xjin_AB_P1R2c,xjin_AB_P1R3a,xjin_AB_P1R3b,xjin_AB_P1R3c,xjin_AB_P2R1a,xjin_AB_P2R1b,xjin_AB_P2R1c,xjin_AB_P2R2a,xjin_AB_P2R2b,xjin_AB_P2R2c,xjin_AB_P2R3a,xjin_AB_P2R3b,xjin_AB_P2R3c,xjin_AB_P3R1a,xjin_AB_P3R1b,xjin_AB_P3R1c,xjin_AB_P3R2a,xjin_AB_P3R2b,xjin_AB_P3R2c,xjin_AB_P3R3a,xjin_AB_P3R3b,xjin_AB_P3R3c,xjin_AB_P4R1a,xjin_AB_P4R1b,xjin_AB_P4R1c,xjin_AB_P4R2a,xjin_AB_P4R2b,xjin_AB_P4R2c,xjin_AB_P4R3a,xjin_AB_P4R3b,xjin_AB_P4R3c,xjin_AB_P5R1a,xjin_AB_P5R1b,xjin_AB_P5R1c,xjin_AB_P5R2a,xjin_AB_P5R2b,xjin_AB_P5R2c,xjin_AB_P5R3a,xjin_AB_P5R3b,xjin_AB_P5R3c,xjin_AS_P0R1a,xjin_AS_P0R1b,xjin_AS_P0R1c,xjin_AS_P0R2a,xjin_AS_P0R2b,xjin_AS_P0R2c,xjin_AS_P0R3a,xjin_AS_P0R3b,xjin_AS_P0R3c,xjin_AS_P1R1a,xjin_AS_P1R1b,xjin_AS_P1R1c,xjin_AS_P1R2a,xjin_AS_P1R2b,xjin_AS_P1R2c,xjin_AS_P1R3a,xjin_AS_P1R3b,xjin_AS_P1R3c,xjin_AS_P2R1a,xjin_AS_P2R1b,xjin_AS_P2R1c,xjin_AS_P2R2a,xjin_AS_P2R2b,xjin_AS_P2R2c,xjin_AS_P2R3a,xjin_AS_P2R3b,xjin_AS_P2R3c,xjin_AS_P3R1a,xjin_AS_P3R1b,xjin_AS_P3R1c,xjin_AS_P3R2a,xjin_AS_P3R2b,xjin_AS_P3R2c,xjin_AS_P3R3a,xjin_AS_P3R3b,xjin_AS_P3R3c,xjin_AS_P4R1a,xjin_AS_P4R1b,xjin_AS_P4R1c,xjin_AS_P4R2a,xjin_AS_P4R2b,xjin_AS_P4R2c,xjin_AS_P4R3a,xjin_AS_P4R3b,xjin_AS_P4R3c,xjin_AS_P5R1a,xjin_AS_P5R1b,xjin_AS_P5R1c,xjin_AS_P5R2a,xjin_AS_P5R2b,xjin_AS_P5R2c,xjin_AS_P5R3a,xjin_AS_P5R3b,xjin_AS_P5R3c,xjin_Flex_Innoculum_Rep1,xjin_Flex_Innoculum_Rep2,xjin_KAPA_Innoculum_Rep1,xjin_KAPA_Innoculum_Rep2,xjin_MB_P0R1a,xjin_MB_P0R1b,xjin_MB_P0R1c,xjin_MB_P0R2a,xjin_MB_P0R2b,xjin_MB_P0R2c,xjin_MB_P0R3a,xjin_MB_P0R3b,xjin_MB_P0R3c,xjin_MB_P1R1a,xjin_MB_P1R1b,xjin_MB_P1R1c,xjin_MB_P1R2a,xjin_MB_P1R2b,xjin_MB_P1R2c,xjin_MB_P1R3a,xjin_MB_P1R3b,xjin_MB_P1R3c,xjin_MB_P2R1a,xjin_MB_P2R1b,xjin_MB_P2R1c,xjin_MB_P2R2a,xjin_MB_P2R2b,xjin_MB_P2R2c,xjin_MB_P2R3a,xjin_MB_P2R3b,xjin_MB_P2R3c,xjin_MB_P3R1a,xjin_MB_P3R1b,xjin_MB_P3R1c,xjin_MB_P3R2a,xjin_MB_P3R2b,xjin_MB_P3R2c,xjin_MB_P3R3a,xjin_MB_P3R3b,xjin_MB_P3R3c,xjin_MB_P4R1a,xjin_MB_P4R1b,xjin_MB_P4R1c,xjin_MB_P4R2a,xjin_MB_P4R2b,xjin_MB_P4R2c,xjin_MB_P4R3a,xjin_MB_P4R3b,xjin_MB_P4R3c,xjin_MB_P5R1a,xjin_MB_P5R1b,xjin_MB_P5R1c,xjin_MB_P5R2a,xjin_MB_P5R2b,xjin_MB_P5R2c,xjin_MB_P5R3a,xjin_MB_P5R3b,xjin_MB_P5R3c,xjin_MS_P0R1a,xjin_MS_P0R1b,xjin_MS_P0R1c,xjin_MS_P0R2a,xjin_MS_P0R2b,xjin_MS_P0R2c,xjin_MS_P0R3a,xjin_MS_P0R3b,xjin_MS_P0R3c,xjin_MS_P1R1a,xjin_MS_P1R1b,xjin_MS_P1R1c,xjin_MS_P1R2a,xjin_MS_P1R2b,xjin_MS_P1R2c,xjin_MS_P1R3a,xjin_MS_P1R3b,xjin_MS_P1R3c,xjin_MS_P2R1a,xjin_MS_P2R1b,xjin_MS_P2R1c,xjin_MS_P2R2a,xjin_MS_P2R2b,xjin_MS_P2R2c,xjin_MS_P2R3a,xjin_MS_P2R3b,xjin_MS_P2R3c,xjin_MS_P3R1a,xjin_MS_P3R1b,xjin_MS_P3R1c,xjin_MS_P3R2a,xjin_MS_P3R2b,xjin_MS_P3R2c,xjin_MS_P3R3a,xjin_MS_P3R3b,xjin_MS_P3R3c,xjin_MS_P4R1a,xjin_MS_P4R1b,xjin_MS_P4R1c,xjin_MS_P4R2a,xjin_MS_P4R2b,xjin_MS_P4R2c,xjin_MS_P4R3a,xjin_MS_P4R3b,xjin_MS_P4R3c,xjin_MS_P5R1a,xjin_MS_P5R1b,xjin_MS_P5R1c,xjin_MS_P5R2a,xjin_MS_P5R2b,xjin_MS_P5R2c,xjin_MS_P5R3a,xjin_MS_P5R3b,xjin_MS_P5R3c,xjin_N_P0R1a,xjin_N_P0R1b,xjin_N_P0R1c,xjin_N_P0R2a,xjin_N_P0R2b,xjin_N_P0R2c,xjin_N_P0R3a,xjin_N_P0R3b,xjin_N_P0R3c,xjin_N_P1R1a,xjin_N_P1R1b,xjin_N_P1R1c,xjin_N_P1R2a,xjin_N_P1R2b,xjin_N_P1R2c,xjin_N_P1R3a,xjin_N_P1R3b,xjin_N_P1R3c,xjin_N_P2R1a,xjin_N_P2R1b,xjin_N_P2R1c,xjin_N_P2R2a,xjin_N_P2R2b,xjin_N_P2R2c,xjin_N_P2R3a,xjin_N_P2R3b,xjin_N_P2R3c,xjin_N_P3R1a,xjin_N_P3R1b,xjin_N_P3R1c,xjin_N_P3R2a,xjin_N_P3R2b,xjin_N_P3R2c,xjin_N_P3R3a,xjin_N_P3R3b,xjin_N_P3R3c,xjin_N_P4R1a,xjin_N_P4R1b,xjin_N_P4R1c,xjin_N_P4R2a,xjin_N_P4R2b,xjin_N_P4R2c,xjin_N_P4R3a,xjin_N_P4R3b,xjin_N_P4R3c,xjin_N_P5R1a,xjin_N_P5R1b,xjin_N_P5R1c,xjin_N_P5R2a,xjin_N_P5R2b,xjin_N_P5R2c,xjin_N_P5R3a,xjin_N_P5R3b,xjin_N_P5R3c,xjin_XT_Innoculum_Rep1,xjin_XT_Innoculum_Rep2".split(",")

values_matrix = []
unitig_list = []

with open('/tmp/bsmith/tmp.8b7PjrUy1c') as f:
    for line in tqdm(f):
        unitig_id, *values = line.split()
        unitig_list.append(int(unitig_id))
        values_matrix.append(np.array(values).astype(float))

values_matrix = np.stack(values_matrix)

result = xr.DataArray(values_matrix, coords=dict(unitig=unitig_list, sample=sample_list))
result

22058444it [56:50, 6468.63it/s]


<xarray.DataArray (unitig: 22058444, sample: 276)> Size: 49GB
array([[2.50000000e+00, 2.00000000e+00, 1.00000000e+00, ...,
        5.25000000e+01, 2.00000000e+00, 1.00000000e+00],
       [6.00000000e+00, 7.86956522e+00, 5.21739130e+00, ...,
        4.13043478e+00, 5.43478261e+00, 1.82608696e+00],
       [4.30769231e+00, 0.00000000e+00, 2.30769231e-01, ...,
        8.53846154e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.95664740e-01, 1.39547206e-01, 1.71274888e-01, ...,
        1.21740527e-01, 2.88391137e-01, 6.36319846e-02],
       [1.04224064e-02, 1.11527955e-01, 2.31661137e-02, ...,
        0.00000000e+00, 2.31719756e+00, 3.20855364e-01],
       [3.58083482e-02, 2.72708328e-02, 4.09694325e-02, ...,
        3.99052914e-04, 4.95151507e-01, 7.02510486e-02]])
Coordinates:
  * unitig   (unitig) int64 176MB 0 1 2 3 ... 22058441 22058442 22058443
  * sample   (sample) <U24 26kB 'xjin_AB_P0R1a' ... 'xjin_XT_Innoculum_Rep2'

In [55]:
result.to_netcdf('data/group/xjin/r.proc.ggcat-k111-withmegahit2-droptips.unitig_depth.nc')

In [24]:
d = pd.read_csv('raw/PRJNA701961_metadata.csv')
d.columns
(d['Sample Name'] == d['Library Name']).all()

True

In [31]:
d = pd.read_csv('raw/PRJNA701961_metadata.csv')

d.rename(columns={'Run': 'sra_accession'}).assign(mgen_id=lambda x: 'watson_' + x['Sample Name'], preprocessing='hfilt.dedup.deadapt.qtrim', r1_path='', r2_path='')[['mgen_id', 'preprocessing', 'sra_accession', 'r1_path', 'r2_path']].set_index('mgen_id').to_csv('test.tsv', sep='\t')

In [6]:
k = 111
n_samples = 2

In [19]:
xr.load_dataarray('data/group/xjin_test2/r.proc.ggcat-k77-withmegahit2-droptips.unitig_depth.nc')

<xarray.DataArray (unitig: 819318, sample: 7)> Size: 46MB
array([[  0.        ,   3.        ,   0.        , ...,   1.        ,
          0.        ,   0.        ],
       [ 77.42857143,  69.42857143,  27.14285714, ...,  41.        ,
          2.        ,   1.14285714],
       [139.28571429,  99.46428571, 119.14285714, ..., 142.32142857,
          1.25      ,   0.        ],
       ...,
       [276.71428571, 191.14285714, 228.71428571, ..., 250.71428571,
          0.        ,   0.        ],
       [275.        , 191.        , 221.        , ..., 247.        ,
          0.        ,   0.        ],
       [  0.36      ,   2.04      ,   1.        , ...,   1.        ,
          0.        ,   0.        ]])
Coordinates:
  * unitig   (unitig) int64 7MB 0 1 10 100 1000 ... 99996 99997 99998 99999
  * sample   (sample) <U22 616B 'xjin_N_P5R2b' ... 'xjin_XT_Innoculum_Rep2'

In [18]:
xr.load_dataarray('data/group/xjin_test2/r.proc.ggcat-k111-withmegahit2-droptips.unitig_depth.nc')

<xarray.DataArray (unitig: 199485, sample: 7)> Size: 11MB
array([[3.36666667e+01, 2.18333333e+01, 1.43333333e+01, ...,
        8.83333333e+00, 0.00000000e+00, 0.00000000e+00],
       [9.78947368e+00, 5.73684211e+00, 1.03684211e+01, ...,
        4.89473684e+00, 3.31578947e+00, 0.00000000e+00],
       [1.00967742e+01, 2.75806452e+00, 1.90322581e+00, ...,
        5.93548387e+00, 7.90322581e-01, 7.41935484e-01],
       ...,
       [4.07916667e+01, 3.48229167e+01, 1.85104167e+01, ...,
        1.50104167e+01, 1.07291667e+00, 7.50000000e-01],
       [3.59105263e+01, 3.99684211e+01, 1.71210526e+01, ...,
        1.64421053e+01, 1.77894737e+00, 4.31578947e-01],
       [1.09909910e+00, 5.13513514e+00, 4.19819820e+00, ...,
        4.03603604e+00, 2.70270270e-01, 3.60360360e-02]])
Coordinates:
  * unitig   (unitig) int64 2MB 10000 10001 10002 10003 ... 196997 196998 196999
  * sample   (sample) <U22 616B 'xjin_N_P5R2b' ... 'xjin_XT_Innoculum_Rep2'

In [7]:
with open("data/group/xjin_test0/r.proc.ggcat-k111-withmegahit2-droptips.fn") as f:
    _, _, _, unitig_sequences  = sz.io.parse_linked_fasta(f, k=k, header_tokenizer=sz.io.ggcat_header_tokenizer, verbose=True)

97138it [00:00, 239860.06it/s]


In [8]:
kmer_to_unitig_id = {}
total_kmers = 0
for unitig_id, sequence in tqdm(unitig_sequences.items()):
    for kmer in sz.sequence.iter_kmers(sequence, k=k):
        kmer_to_unitig_id[kmer] = unitig_id
        total_kmers += 1

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97138/97138 [01:36<00:00, 1002.80it/s]


In [25]:
total_kmers

132146899

In [51]:
unitig_to_mean_depth = xr.load_dataarray('data/group/xjin_test0/r.proc.ggcat-k111-withmegahit2-droptips.unitig_depth.nc')
unitig_to_mean_depth

<xarray.DataArray (unitig_id: 97138, sample_id: 2)> Size: 2MB
array([[ 9.8       ,  8.        ],
       [ 5.        ,  6.        ],
       [ 4.27272727,  1.42424242],
       ...,
       [10.25310725,  4.72754959],
       [13.64293223,  3.44288074],
       [ 1.38783876,  0.65333895]])
Coordinates:
  * unitig_id  (unitig_id) <U5 2MB '0' '1' '2' '3' ... '97135' '97136' '97137'
  * sample_id  (sample_id) <U13 104B 'xjin_AB_P0R1a' 'xjin_AB_P0R1b'

In [53]:
unitig_to_mean_depth2

<xarray.DataArray (unitig: 199485, sample: 7)> Size: 11MB
array([[3.36666667e+01, 2.18333333e+01, 1.43333333e+01, ...,
        8.83333333e+00, 0.00000000e+00, 0.00000000e+00],
       [9.78947368e+00, 5.73684211e+00, 1.03684211e+01, ...,
        4.89473684e+00, 3.31578947e+00, 0.00000000e+00],
       [1.00967742e+01, 2.75806452e+00, 1.90322581e+00, ...,
        5.93548387e+00, 7.90322581e-01, 7.41935484e-01],
       ...,
       [4.07916667e+01, 3.48229167e+01, 1.85104167e+01, ...,
        1.50104167e+01, 1.07291667e+00, 7.50000000e-01],
       [3.59105263e+01, 3.99684211e+01, 1.71210526e+01, ...,
        1.64421053e+01, 1.77894737e+00, 4.31578947e-01],
       [1.09909910e+00, 5.13513514e+00, 4.19819820e+00, ...,
        4.03603604e+00, 2.70270270e-01, 3.60360360e-02]])
Coordinates:
  * unitig   (unitig) int64 2MB 10000 10001 10002 10003 ... 196997 196998 196999
  * sample   (sample) <U22 616B 'xjin_N_P5R2b' ... 'xjin_XT_Innoculum_Rep2'

In [52]:
unitig_to_mean_depth2 = xr.load_dataarray('data/group/xjin_test2/r.proc.ggcat-k111-withmegahit2-droptips.unitig_depth.nc')

In [26]:
unitig_to_unparsed_counts = defaultdict(list)
with open("data/group/xjin_test0/r.proc.kmc-k111-withmegahit2-droptips.kcounts_merged.tsv") as f:
    # Skip header
    next(f)
    for line in tqdm(f):
        kmer, *counts = line.split('\t')
        if kmer not in kmer_to_unitig_id:  # Not canonical
            kmer = sz.io.reverse_complement(kmer)
        unitig_to_unparsed_counts[kmer_to_unitig_id[kmer]].append(counts)

108718572it [15:44, 115157.15it/s]


In [38]:
unitig_to_mean_depth = {}

for unitig_id, sequence in tqdm(unitig_sequences.items()):
    unparsed_matrix = unitig_to_unparsed_counts[unitig_id]
    # assert len(unparsed_matrix) == len(sequence) - k + 1, "Incorrect number of kmers!!"
    matrix = np.zeros((len(sequence) - k + 1, n_samples))
    for i, row in enumerate(unparsed_matrix):
        matrix[i] = [int(token.strip()) for token in row]
    unitig_to_mean_depth[unitig_id] = matrix.mean(0)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97138/97138 [02:43<00:00, 592.80it/s]


In [39]:
total_with_depth = 0
for i in unitig_to_unparsed_counts:
    if unitig_to_unparsed_counts[i]:
        total_with_depth += 1

total_with_depth

96236

In [48]:
(pd.DataFrame(unitig_to_mean_depth.values(), index=unitig_to_mean_depth.keys()).sum(1) == 0).mean()

0.009285758405567337

In [37]:
len(unitig_to_unparsed_counts[i])

18405

In [34]:
kmer_to_unitig_id[unitig_sequences[i]]

KeyError: 'GTACCAGGTGAAACGTTCCATTGAAGGGGACGCGGCCCGGTAAGGAAAAGTATGGCTGGACAAATCACAACATAACGTTTATAATAATGATAATCAAAAAAAAAAAAAAACACAAAACCTTTTGGCAAACCTGGAGAAATCCAGCGACGCAAAGCTAACAGGGCCTGTAAAATGGCAGCCAGCTGCACAGTGTTTTTCCTCCGCCCTGAAAAATGCGGAATGGTGAGAACCGTCAGCCGTCTGTCTATGGGACAGGCGGCTTTTCGTTGTTTTGCCATCCAATAAGTAAGGAGAAGTTGCATTGTGCGAAGATTAATCCGAAGTCTACAGGCCCAGTTTCTGGTGCTGATCCTGGGGATCACCCTGGCCCTGATCCTGGTGCTGGGCTTTTTGAACCTGCAGACCGTCAACGGTACCTCCACCCGGATTGCCGCCATGAGCCTGAACTGGCAGGCCCAGCGGACTTCCGCCCCCATCCAGGGGGTGCTCACCAGCACCATGGACATGACGGACATCCTGGCCAATGGATTGATCAAAGATGTACCGGACCCCGGCCGGCTGAGTGATCCGGCTTTCCGGAAAGGTATCCTGGTCCATCTCCAGAAGCATTTCAACCTGTCCGCCAGGAATTCCACCACGGTCTTTGGCTATTATGTCCAGTTCAATCCGGAGCTGACCGGAGGGCCTGCCGGGTTCTGGTACACCTGGGATCCCCGGCGGAGCCAGTACATTTCCCATTCCCGGGAAGCCATGGGGGCCTATTTCTATGGAAAGGGCGGTTTCCTCCCCTATTTCCGGGATCTGATTGCCGATGGAAAACCGGTCTGGCGGAACCCCTATGACGACCACAACTTCGGGGTCCGGCGGATTTCCTATATCGTTCCGGTGTACAGCAACGGCACCTTCGTGGCCTTCCTGGGCATGAGCATCCGGATGGATACCCTGGTGACCATGATGTCCCACGCCCGGATCTACGACACCGGCTATGC

In [29]:
len(unitig_sequences[i])

111

In [30]:
unitig_to_unparsed_counts[i]

[['660', '362\n']]

In [ ]:
len(sequence)

In [ ]:
len(unitig_sequences['4'])

In [ ]:
len(unitig_to_unparsed_counts['4'])

In [ ]:
unparsed_matrix

In [ ]:
sequence

In [ ]:
unitig_to_mean_depth

In [ ]:
mean_depth_table = pd.DataFrame(unitig_to_mean_depth.values(), index=unitig_to_mean_depth.keys())

In [ ]:
mean_depth_table

In [ ]:
x = list(unitig_to_counts.keys())
x[100]

In [ ]:
len(unitig_sequences['1000002'])

In [ ]:
unitig_to_counts['1000002'].mean(0)

In [ ]:
kmtricks_depth = xr.load_dataarray('data/group/xjin_test0/r.proc.ggcat-k111-denovo-kmtricks.unitig_depth.nc')

In [ ]:
kmc_masked_depth = xr.load_dataarray('data/group/xjin_test0/r.proc.ggcat-k111-denovo-masked.unitig_depth.nc')

In [ ]:
kmc_graph = sz.io.load_graph('data/group/xjin_test0/r.proc.ggcat-k111-denovo-masked.sz')

In [ ]:
vertex_to_unitig = pd.Series([int(s[:-1]) for s in kmc_graph.vp['sequence']], index=kmc_graph.get_vertices())
long_tip_vertices = sz.topology.find_tips(kmc_graph, also_required=(kmc_graph.vp['length'].a > 222))
short_tip_vertices = sz.topology.find_tips(kmc_graph, also_required=(kmc_graph.vp['length'].a < 222))
long_tip_unitigs = list(set(vertex_to_unitig.loc[long_tip_vertices].values))
short_tip_unitigs = list(set(vertex_to_unitig.loc[short_tip_vertices].values))

kmc_masked_depth.loc[short_tip_unitigs].values.sum()  # 0
(kmc_masked_depth.loc[long_tip_unitigs].sum('sample') == 0).sum()  # 0
(kmc_masked_depth.sel(unitig=vertex_to_unitig.drop(sz.topology.find_tips(kmc_graph)).values).sum("sample") == 0).sum()  # 0

In [ ]:
low_depth_unitigs = idxwhere((kmc_masked_depth.sum("sample") == 2).to_series())

In [ ]:
((kmc_masked_depth < kmtricks_depth)).sum()

In [ ]:
kmtricks_depth.sel(unitig=((kmc_masked_depth > kmtricks_depth)).any('sample'))

In [ ]:
kmc_masked_depth.sel(unitig=((kmc_masked_depth < kmtricks_depth)).any('sample')).sum()

In [ ]:
kmc_masked_depth.unitig[((kmc_masked_depth < kmtricks_depth)).any('sample')].to_series()

In [ ]:
(kmc_masked_depth.sel(unitig=vertex_to_unitig.drop(sz.topology.find_tips(kmc_graph)).values).sum("sample") < 1).sum()  # 0

In [ ]:
(kmtricks_depth.sel(unitig=vertex_to_unitig.drop(sz.topology.find_tips(kmc_graph)).values).sum("sample") < 100).sum()  # 0

In [ ]:
kmtricks_depth.sel(unitig=2279938)

In [ ]:
kmc_masked_depth.sel(unitig=2279938)

In [ ]:
kmc_masked_depth.sel(unitig=low_depth_unitigs)

In [ ]:
(kmtricks_depth.sum("sample") == 0).sum()

In [ ]:
kmtricks_depth.sel(unitig=low_depth_unitigs).values.sum()

In [ ]:
kmc_masked_depth.sel(unitig=14000)

In [ ]:
kmc_graph.vp['sequence'][0]

In [ ]:
kmtricks_depth.sel(unitig=572824)

In [ ]:
kmc_masked_depth.sel(unitig=572824)

In [ ]:
sz.topology.find_tips(kmc_graph)

In [ ]:
kmtricks_depth.sel(unitig=1098)

In [ ]:
kmc_masked_depth.sel(unitig=1098)

In [ ]:
mpl.rcParams['figure.dpi'] = 200
# sns.set_context('talk')

In [ ]:
# Plotting parameters

length_bins = np.logspace(0, 6.5, num=51)
depth_bins = np.logspace(-1, 4, num=51)

draw_graphs = True
run_number = 48  # Label for output files/figures

In [ ]:
group = 'xjin'
graph_type = 'withmegahit'
deconv = 'norm-10-10'
kmtricks = 'k111-m3-r2'
clust_params = 'e50-d20'

In [ ]:
# Metadata
sample_idx_to_id = pd.read_table('meta/mgen_group.tsv', names=['mgen_id', 'mgen_group'])[lambda x: x.mgen_group == group].mgen_id.to_frame().assign(idx=lambda x: range(len(x))).set_index('idx').mgen_id
sample_idx_to_id

# NOTE: This only applies to xjin samples:
sample_idx_to_id = sample_idx_to_id.str.split('_', n=1).str[1]
sample_idx_to_id.head()

In [ ]:
clust_meta = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.meta.tsv', index_col='cluster')
vertex_to_clust = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.vertex.tsv', index_col='vertex').cluster
segement_x_clust = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.segment.tsv', index_col='segment').cluster
print(len(clust_meta))
clust_meta.sort_values('total_length', ascending=False).head(5)

In [ ]:
%%time

unpressed_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2-unpressed.sz')
unpressed_results = sz.results.extract_vertex_data(unpressed_graph)
print((unpressed_results.length * unpressed_results.total_depth).sum() / unpressed_results.total_depth.sum())

obs_segment_depth = sz.results.extract_segment_depth(unpressed_graph)

In [ ]:
segment_length = unpressed_results.assign(segment=lambda x: x.segments.str[0]).set_index('segment').length
# For each segment how many extra paths could have gone through it? max((N * M) - 1, 0)
segment_complexity = pd.Series((unpressed_results.num_in_neighbors * unpressed_results.num_out_neighbors - 1).where(lambda x: x>0, 0).values, index=unpressed_results.segments.str[0])

plt.hist(segment_complexity, bins=np.arange(15))
plt.yscale('log')

In [ ]:
%%time
final_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.sz')
final_results = sz.results.extract_vertex_data(final_graph)
vertex_complexity = final_results.segments.explode().to_frame('segment').reset_index().join(segment_complexity.rename('segment_complexity'), on='segment').drop_duplicates().groupby('vertex').segment_complexity.sum()

final_results = final_results.assign(
    complexity=vertex_complexity,
    clust=vertex_to_clust.reindex(final_results.index, fill_value=-1),
    
)

vertex_depth = sz.results.depth_table(final_graph, final_graph.get_vertices()).T
vertex_length = pd.Series(final_graph.vp['length'], index=final_graph.get_vertices())

In [ ]:
# Normalize by rpoB and gyrA depth

marker_model = 'TIGR02013'

marker_genes = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.cds.tran.hmmer-{marker_model}-ga.tsv', names=['orf', 'gene_name', 'bitscore']).assign(vertex=lambda x: x.orf.str.split('_').str[0].astype(int))
_marker_gene_vertex_list = list(set(marker_genes.vertex))
marker_depth = vertex_depth.loc[_marker_gene_vertex_list].T
total_marker_depth = marker_depth.sum(1)

normalized_vertex_depth = vertex_depth.divide(total_marker_depth)

marker_rabund = marker_depth.divide(total_marker_depth, axis=0)
# plt.hist(marker_depth.sum())
sns.clustermap(marker_rabund.rename(sample_idx_to_id) + 1e-5, norm=mpl.colors.SymLogNorm(1e-5), metric='cosine')

In [ ]:
kraken_read_counts = pd.read_csv('/pollard/home/xiaofanj/microbiomeAdhesion/intermediates/biofilmBeadExpV2/customKrakenOutputs/customKraken2BrackenAbundances.csv').set_index(['Strain_Name', 'sample']).new_est_reads.unstack('Strain_Name', fill_value=0)
kraken_rabund = kraken_read_counts.divide(kraken_read_counts.sum(1), axis=0)

sns.clustermap(kraken_rabund, norm=mpl.colors.SymLogNorm(1e-5), metric='cosine')

In [ ]:
%%time

print((final_results.length * final_results.total_depth).sum() / final_results.total_depth.sum())
final_segment_depth = sz.results.extract_segment_depth(final_graph)

In [ ]:
# Used below for visualizing the graph
# This part is slow enough that it's better to cache it and pass it to the get_shortest_distance function.
backlinked_unpressed_graph = sz.topology.backlinked_graph(unpressed_graph)

In [ ]:
%%time

y, x = obs_segment_depth.align(final_segment_depth, join='left')

bins = np.concatenate([[0], np.logspace(-3, 7, num=50)])

fig = plt.figure()
plt.hist2d(x.values.ravel(), y.values.ravel(), bins=bins, norm=mpl.colors.SymLogNorm(linthresh=1))
plt.xscale('symlog', linthresh=1e-3, linscale=0.2)
plt.yscale('symlog', linthresh=1e-3, linscale=0.2)
plt.plot([1e-2, 1e5], [1e-2, 1e5], lw=1, linestyle='--', color='r')
plt.colorbar()
plt.xticks(np.logspace(-3, 7, num=6))
plt.yticks(np.logspace(-3, 7, num=6))
plt.xlabel('Predicted Depth')
plt.ylabel('Observed Depth')

print(sp.stats.pearsonr(x.values.ravel(), y.values.ravel()))

fig = plt.figure()
plt.scatter(x.sum(), y.sum())

In [ ]:
sp.stats.pearsonr(x.values.ravel(), y.values.ravel()).statistic ** 2

In [ ]:
notips_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.sz')
notips_results = sz.results.extract_vertex_data(notips_graph)
sz.stats.depth_weighted_mean_tig_length(notips_graph)

In [ ]:
notips_unsmoothed_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.sz')
notips_unsmoothed_results = sz.results.extract_vertex_data(notips_unsmoothed_graph)
sz.stats.depth_weighted_mean_tig_length(notips_unsmoothed_graph)

In [ ]:
y, x = notips_results.total_depth.align(notips_unsmoothed_results.total_depth)

bins = np.concatenate([[0], np.logspace(-3, 7, num=50)])

fig = plt.figure()
plt.hist2d(x, y, bins=bins, norm=mpl.colors.SymLogNorm(linthresh=1))
plt.xscale('symlog', linthresh=1e-3, linscale=0.2)
plt.yscale('symlog', linthresh=1e-3, linscale=0.2)
plt.plot([1e-2, 1e5], [1e-2, 1e5], lw=1, linestyle='--', color='r')
plt.colorbar()
plt.xticks(np.logspace(-3, 7, num=6))
plt.yticks(np.logspace(-3, 7, num=6))
plt.xlabel('Unsmoothed')
plt.ylabel('Smoothed')

sp.stats.pearsonr(x, y)

In [ ]:
round0_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_0.sz')
round0_results = sz.results.extract_vertex_data(round0_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round0_graph))

round1_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_1.sz')
round1_results = sz.results.extract_vertex_data(round1_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round1_graph))

round2_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_2.sz')
round2_results = sz.results.extract_vertex_data(round2_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round2_graph))

round3_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_3.sz')
round3_results = sz.results.extract_vertex_data(round3_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round3_graph))

round4_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_4.sz')
round4_results = sz.results.extract_vertex_data(round4_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round4_graph))

round5_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_5.sz')
round5_results = sz.results.extract_vertex_data(round5_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round5_graph))

round6_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_6.sz')
round6_results = sz.results.extract_vertex_data(round6_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round6_graph))

round7_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_7.sz')
round7_results = sz.results.extract_vertex_data(round7_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round7_graph))

round8_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_8.sz')
round8_results = sz.results.extract_vertex_data(round8_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round8_graph))

round9_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.checkpoints.d/checkpoint_9.sz')
round9_results = sz.results.extract_vertex_data(round9_graph)
# print(sz.stats.depth_weighted_mean_tig_length(round9_graph))

In [ ]:
# Overall, we get about 2x as much long sequence depth (distribution of tig lengths of assigned kmers)
fig, ax = plt.subplots(figsize=(10, 5))

_results_mapping = {
    'raw': unpressed_results,
    # 'notips-unsmoothed': unsmoothed_results,
    'notips': notips_results,
    # 'safeonly': safeonly_results,
    'round1': round1_results,
    'round2': round2_results,
    'round3': round3_results,
    'round4': round4_results,
    # 'round10': round10_results,
    # 'unsmoothed': unsmoothed_results,
    # 'round15': round15_results,
    # 'round25': round25_results,
    'final': final_results,
    # 'norefs': norefs_results,
    # 'alt1': alt1_results,
    # 'alt3': alt3_results,
    # 'alt4': alt4_results,
    # 'alt5': alt5_results,
}

_cmap = lib.plot.construct_ordered_palette(_results_mapping.keys(), cm='rainbow')


for _label in _results_mapping:
    _results = _results_mapping[_label]
    _c = _cmap[_label]
    d = _results.sort_values('length', ascending=False).assign(length_x_depth=lambda x: x.total_depth * x.length, length_x_depth_cumsum=lambda x: x.length_x_depth.cumsum())
    ax.plot('length', 'length_x_depth_cumsum', data=d, label=_label, color=_c, lw=2)

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Cumulative K-mers Assigned')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(left=1e3)
# ax.set_ylim(1e9, 4e10)

ax.legend(loc='lower left') # bbox_to_anchor=(1, 1))
# ax.invert_xaxis()

In [ ]:
_results_mapping = {
    'cdBG': unpressed_results,
    'Trim Tips': notips_results,
    'Drop Low Depth': round0_results,
    'Round 1': round1_results,
    'Round 2': round2_results,
    'Round 3': round3_results,
    'Round 4': round4_results,
    'Round 5': round5_results,
    'Round 6': round6_results,
    'Round 7': round7_results,
    'Round 8': round8_results,
    'Round 9': round9_results,
    'Round 10': final_results,
}

In [ ]:
depth_weighted_median_tig_length_data = {'Kmer': 1}
for k in _results_mapping:
    d0 = _results_mapping[k].assign(total_kmers=lambda x: x.length * x.total_depth).sort_values('length', ascending=False)
    median_vertex = ((d0.total_kmers.cumsum() / d0.total_kmers.sum()) > 0.5).idxmax()
    depth_weighted_median_tig_length_data[k] = d0.loc[median_vertex].length
    print(f'{k}: {depth_weighted_median_tig_length_data[k]}')
    
depth_weighted_median_tig_length_data = pd.Series(depth_weighted_median_tig_length_data)

In [ ]:
_color_palette = lib.plot.construct_ordered_palette(depth_weighted_median_tig_length_data.keys())
plt.plot(depth_weighted_median_tig_length_data)
plt.scatter(
    range(len(depth_weighted_median_tig_length_data)),
    depth_weighted_median_tig_length_data.values,
    c=depth_weighted_median_tig_length_data.index.to_series().map(_color_palette),
    edgecolor='grey',
    zorder=2,
)
for x, (_label, y) in enumerate(depth_weighted_median_tig_length_data.items()):
    plt.annotate(y, xy=(x, y), rotation=-45, ha='left', va='top', xytext=(5, -5), textcoords='offset pixels')
lib.plot.rotate_xticklabels(rotation=-45, ha='left', va='top')
plt.ylabel('Kmer-weighted median path length')
plt.ylim(bottom=-3000)
plt.xlim(right=14)

In [ ]:
final_results.sort_values(['num_segments'], ascending=False).head()

In [ ]:
final_results.sort_values(['length'], ascending=False).head(10)

In [ ]:
final_results.sort_values(['complexity'], ascending=False).head()

In [ ]:
final_results[lambda x: (x.complexity > x.num_segments * 1.2) & (x.num_segments > 50) & (x.num_segments < 100) & (x.total_depth > 500)]

In [ ]:
final_results.segments.explode().value_counts().head()

In [ ]:
# Use this to find tigs of interest.
# WORKHERE

final_results.loc[sz.results.iter_find_vertices_with_any_segment(final_graph, ['1283360+', '1283360-'])].sort_values(['num_segments', 'total_depth'], ascending=False)

In [ ]:
# Select paths that share any vertices with a given path.
focal_path = 210689
focal_segments = list(set(final_results.loc[focal_path].segments))
related_paths = list(sz.results.iter_find_vertices_with_any_segment(final_graph, final_results.loc[focal_path].segments))

# # Select paths going through a single segment
# focal_segments = ['1668157+'] # Single focal segment
# related_paths = list(sz.results.iter_find_vertices_with_any_segment(final_graph, focal_segments))

related_segments = list(set(chain(*final_results.loc[related_paths].segments)))
print(len(related_paths), len(related_segments))

# unpressed_graph_focal_vertices = list(sz.results.iter_find_vertices_with_any_segment(unpressed_graph, focal_segments))
unpressed_graph_core_vertices = list(sz.results.iter_find_vertices_with_any_segment(unpressed_graph, related_segments))
pressed_graph_core_vertices = list(sz.results.iter_find_vertices_with_any_segment(notips_graph, related_segments))

# in_path = unpressed_graph.new_vertex_property('bool', vals=pd.Series(unpressed_graph.get_vertices()).isin(unpressed_graph_focal_vertices).values)
# path_graph = gt.GraphView(unpressed_graph, vfilt=in_path)
# print(path_graph)

in_core = unpressed_graph.new_vertex_property('bool', vals=pd.Series(unpressed_graph.get_vertices()).isin(unpressed_graph_core_vertices).values)
core_graph = gt.GraphView(unpressed_graph, vfilt=in_core)
print(core_graph)

# in_core_pressed = notips_graph.new_vertex_property('bool', vals=pd.Series(notips_graph.get_vertices()).isin(pressed_graph_core_vertices).values)
# core_graph_pressed = gt.GraphView(notips_graph, vfilt=in_core_pressed)
# print(core_graph_pressed)

final_results.loc[related_paths].sort_values('num_segments', ascending=False)

In [ ]:
%%time
unpressed_graph_distance_to_core = sz.topology.get_shortest_distance_to_any_vertex(unpressed_graph, unpressed_graph_core_vertices, unpressed_graph.vp['length'])

In [ ]:
radius = 10

in_neighborhood = unpressed_graph.new_vertex_property('bool', vals=unpressed_graph_distance_to_core.a <= radius)
neighborhood_graph = gt.GraphView(unpressed_graph, vfilt=in_neighborhood)
neighborhood_graph

In [ ]:
# sample_for_depth = 118

_graph = gt.Graph(neighborhood_graph, prune=True)
# _graph = gt.Graph(core_graph_pressed, prune=True)
# _graph = gt.Graph(path_graph, prune=True)
# _graph = gt.Graph(neighborhood_graph_final, prune=True)

gt.seed_rng(1)
np.random.seed(2)
sz.draw.update_xypositions(_graph, layout=gt.draw.random_layout, shape=(10, 10))  # NOTE: This allows for reproducible layouts.
sz.draw.update_xypositions(_graph, layout=gt.draw.sfdp_layout, p=1.4, K=1)


_offset = 1  # Controls color range of nodes.
vertex_color = _graph.new_vertex_property('float', vals=np.sqrt(sz.results.total_depth_property(_graph).a + _offset))
# vertex_color = _graph.new_vertex_property('float', vals=np.sqrt(_graph.vp['depth'].get_2d_array(pos=[sample_for_depth]) + _offset))


outpath = f'fig/run-{run_number}/neighborhood-{focal_path}.final.png'
sz.draw.draw_graph(
    _graph,
    # vertex_text=_graph.vp['sequence'],
    vertex_text=_graph.vp['length'],
    # vertex_text='',
    # vertex_halo=in_path,
    vertex_font_size=50,
    vertex_fill_color=vertex_color,
    edge_pen_width=15,
    output=outpath,
    vcmap=(mpl.cm.magma, 1),
    output_size=(1500, 1500),
)
print(outpath)

In [ ]:
for path in list(final_results.loc[related_paths].sort_values('num_segments', ascending=False).head(6).index):
    _vertices = list(sz.results.iter_find_vertices_with_any_segment(_graph, final_results.loc[path].segments))
    print(len(_vertices))
    in_path = _graph.new_vertex_property('bool', val=False)
    in_path.a[_vertices] = 1
    outpath = f'fig/run-{run_number}/neighborhood-{focal_path}.final.neighbor-{path}.pdf'
    sz.draw.draw_graph(
        _graph,
        # vertex_text=_graph.vp['sequence'],
        # vertex_text=_graph.vp['length'],
        vertex_text='',
        vertex_halo=in_path,
        vorder=in_path.transform(lambda x: x.astype(float)),
        vertex_font_size=5,
        vertex_fill_color=vertex_color,
        # edge_pen_width=10,
        output=outpath,
        vcmap=(mpl.cm.magma, 1),
        output_size=(400, 400),
    )
    print(outpath)

In [ ]:
radius = 61000

in_neighborhood = unpressed_graph.new_vertex_property('bool', vals=unpressed_graph_distance_to_core.a <= radius)
neighborhood_graph = gt.GraphView(unpressed_graph, vfilt=in_neighborhood)
neighborhood_graph

In [ ]:
# sample_for_depth = 118

_graph = gt.Graph(neighborhood_graph, prune=True)
# _graph = gt.Graph(core_graph_pressed, prune=True)
# _graph = gt.Graph(path_graph, prune=True)
# _graph = gt.Graph(neighborhood_graph_final, prune=True)

gt.seed_rng(1)
np.random.seed(2)
sz.draw.update_xypositions(_graph, layout=gt.draw.random_layout, shape=(10, 10))  # NOTE: This allows for reproducible layouts.
# sz.draw.update_xypositions(_graph, layout=gt.draw.sfdp_layout, p=1.4, K=1)
sz.draw.update_xypositions(_graph, layout=gt.draw.sfdp_layout)


_offset = 1  # Controls color range of nodes.
vertex_color = _graph.new_vertex_property('float', vals=np.log(sz.results.total_depth_property(_graph).a + _offset))
# vertex_color = _graph.new_vertex_property('float', vals=np.sqrt(_graph.vp['depth'].get_2d_array(pos=[sample_for_depth]) + _offset))


outpath = f'fig/run-{run_number}/region-{focal_path}.final.png'
sz.draw.draw_graph(
    _graph,
    # vertex_text=_graph.vp['sequence'],
    # vertex_text=_graph.vp['length'],
    vertex_text='',
    # vertex_halo=in_path,
    # vertex_font_size=50,
    vertex_size=20,
    vertex_fill_color=vertex_color,
    # edge_pen_width=15,
    output=outpath,
    vcmap=(mpl.cm.magma, 1),
    output_size=(1500, 1500),
)
print(outpath)

In [ ]:
# # Select paths that share any vertices with a given path.
# focal_segments = list(set(final_results.loc[focal_path].segments))
# related_paths = list(sz.results.iter_find_vertices_with_any_segment(final_graph, final_results.loc[focal_path].segments))

# # # Select paths going through a single segment
# # focal_segments = ['1668157+'] # Single focal segment
# # related_paths = list(sz.results.iter_find_vertices_with_any_segment(final_graph, focal_segments))

# related_segments = list(set(chain(*final_results.loc[related_paths].segments)))
# print(len(related_paths), len(related_segments))

path_membership = final_results.loc[related_paths].segments.explode().reset_index().value_counts().unstack(fill_value=0)
_obs_segment_depth = obs_segment_depth.loc[path_membership.columns].T
est_path_depth = vertex_depth.T[related_paths]
pred_unitig_depth = final_segment_depth.loc[path_membership.columns].T
resid_unitig_depth = _obs_segment_depth - pred_unitig_depth
relative_resid_unitig_depth = resid_unitig_depth / (pred_unitig_depth + 1)
_segment_length_colors = segment_length.loc[path_membership.columns].map(mpl.cm.viridis)

max_obs = _obs_segment_depth.max().max()
max_resid = np.abs(relative_resid_unitig_depth).max().max()

# unitig_linkage = fastcluster.linkage(_obs_segment_depth.T, metric='euclidean', method='average')
unitig_linkage = fastcluster.linkage(path_membership.T, metric='cosine', method='average')
path_linkage = fastcluster.linkage(path_membership, metric='cosine', method='average')
sample_linkage = fastcluster.linkage(_obs_segment_depth, metric='euclidean', method='average')

sns.clustermap(est_path_depth, row_linkage=sample_linkage, col_linkage=path_linkage, yticklabels=0, xticklabels=1, figsize=(3, 3), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs))
sns.clustermap(est_path_depth.T, col_linkage=sample_linkage, row_linkage=path_linkage, xticklabels=0, yticklabels=1, figsize=(3, 3), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs))
sns.clustermap(path_membership, row_linkage=path_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), xticklabels=0, norm=mpl.colors.SymLogNorm(1, vmin=0))
sns.clustermap(_obs_segment_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs), xticklabels=0)
sns.clustermap(pred_unitig_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs), xticklabels=0)
sns.clustermap(relative_resid_unitig_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=-max_resid, vmax=max_resid), cmap='coolwarm', xticklabels=0)

In [ ]:
# NOTE: This version has a subset of samples for easier visualization.

np.random.seed(0)
_sample_list = np.random.choice(obs_segment_depth.columns, size=10)

path_membership = final_results.loc[related_paths].segments.explode().reset_index().value_counts().unstack(fill_value=0)
_obs_segment_depth = obs_segment_depth.loc[path_membership.columns, _sample_list].T
est_path_depth = vertex_depth.T[related_paths].loc[_sample_list]
pred_unitig_depth = final_segment_depth.loc[path_membership.columns, _sample_list].T
resid_unitig_depth = _obs_segment_depth - pred_unitig_depth
relative_resid_unitig_depth = resid_unitig_depth / (pred_unitig_depth + 1)
_segment_length_colors = segment_length.loc[path_membership.columns].map(mpl.cm.viridis)

max_obs = _obs_segment_depth.max().max()
max_resid = np.abs(relative_resid_unitig_depth).max().max()

# unitig_linkage = fastcluster.linkage(_obs_segment_depth.T, metric='euclidean', method='average')
unitig_linkage = fastcluster.linkage(path_membership.T, metric='cosine', method='average')
path_linkage = fastcluster.linkage(path_membership, metric='cosine', method='average')
sample_linkage = fastcluster.linkage(_obs_segment_depth, metric='euclidean', method='average')

sns.clustermap(est_path_depth, row_linkage=sample_linkage, col_linkage=path_linkage, yticklabels=0, xticklabels=1, figsize=(3, 3), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs))
sns.clustermap(est_path_depth.T, col_linkage=sample_linkage, row_linkage=path_linkage, xticklabels=0, yticklabels=1, figsize=(3, 3), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs))
sns.clustermap(path_membership, row_linkage=path_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), xticklabels=0, norm=mpl.colors.SymLogNorm(1, vmin=0))
sns.clustermap(_obs_segment_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs), xticklabels=0)
sns.clustermap(pred_unitig_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=0, vmax=max_obs), xticklabels=0)
sns.clustermap(relative_resid_unitig_depth, row_linkage=sample_linkage, col_linkage=unitig_linkage, col_colors=_segment_length_colors, figsize=(15, 5), norm=mpl.colors.SymLogNorm(1e-1, vmin=-max_resid, vmax=max_resid), cmap='coolwarm', xticklabels=0)

In [ ]:
_cross_correlation.loc[x_id].idxmax()

In [ ]:
path_order = [184977, 210689, 187913]
_palette = {184977: 'dodgerblue', 210689: 'limegreen', 187913: 'hotpink'}
_rename_strain = {
    'Veillonella-sp-3-1-44': 'Veillonella parvulla Strain A',
    'Veillonella-sp-6-1-27': 'Veillonella parvulla Strain B',
    'Veillonella-dispar-ATCC-17748': 'Veillonella dispar',
}

x = normalized_vertex_depth.loc[path_order]
y = kraken_rabund.T
_cross_correlation = pd.DataFrame(1 - sp.spatial.distance.cdist(x, y, metric='cosine'), index=x.index, columns=y.index)


fig, axs = lib.plot.subplots_grid(ncols=3, naxes=len(path_order), ax_height=3.5)

for x_id, ax in zip(x.index, axs.flatten()):
    y_id = _cross_correlation.loc[x_id].idxmax()
    print(y_id)

    max_val = max(x.loc[x_id].max(), y.loc[y_id].max())

    ax.scatter(x.loc[x_id], y.loc[y_id], s=15, color=_palette[x_id])
    ax.plot([0, 1], [0, 1], color='k', linestyle='--')
    ax.set_xscale('symlog', linthresh=1e-4, linscale=0.25)
    ax.set_yscale('symlog', linthresh=1e-4, linscale=0.25)
    ax.set_xlabel(f'Path {x_id}')
    ax.set_ylabel(_rename_strain[y_id], fontstyle='italic')
    ax.set_aspect(1)
    ax.set_xticks([1e-4, 1e-3, 1e-2, 1e-1, 1e-0])
    # ax.annotate('{:0.01e}'.format(clust_meta.loc[clust_id].total_length), xy=(0.6, 0.3), xycoords='axes fraction')
    # cross_correlation.loc[strain_id].sort_values(ascending=False).head()
    
fig.tight_layout()

In [ ]:
clust_depth = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.depth.tsv').rename(columns={'Unnamed: 0': 'cluster'}).set_index('cluster').T.rename(int)
sns.clustermap(clust_depth.rename(sample_idx_to_id).loc[:, idxwhere(clust_meta.total_length > 1_000_000)], metric='cosine', norm=mpl.colors.SymLogNorm(1))

In [ ]:
clust_rabund = clust_depth.divide(marker_depth.sum(1), axis=0)

sns.clustermap(clust_rabund[idxwhere(clust_meta.total_length > 1_000_000)].rename(sample_idx_to_id), norm=mpl.colors.SymLogNorm(1e-5), metric='cosine')

In [ ]:
x = kraken_rabund
y = clust_rabund[idxwhere(clust_meta.total_length > 2_000_000)].rename(sample_idx_to_id)

x, y = x.align(y, join='inner', axis='index')

cross_correlation = pd.DataFrame(1 - sp.spatial.distance.cdist(x.T, y.T, metric='cosine'), index=x.columns, columns=y.columns)

sns.clustermap(cross_correlation)

In [ ]:
_strain_id_list = kraken_rabund.columns

fig, axs = lib.plot.subplots_grid(ncols=5, naxes=len(_strain_id_list))

for strain_id, ax in zip(_strain_id_list, axs.flatten()):
    clust_id = cross_correlation.loc[strain_id].idxmax()

    max_val = max(x[strain_id].max(), y[clust_id].max())

    ax.scatter(x[strain_id], y[clust_id])
    ax.plot([0, max_val], [0, max_val])
    ax.set_xscale('symlog', linthresh=1e-5)
    ax.set_yscale('symlog', linthresh=1e-5)
    ax.set_xlabel(strain_id)
    ax.set_ylabel(f'bin {clust_id}')
    ax.set_aspect(1)
    ax.annotate('{:0.01e}'.format(clust_meta.loc[clust_id].total_length), xy=(0.6, 0.3), xycoords='axes fraction')
    # cross_correlation.loc[strain_id].sort_values(ascending=False).head()
    
fig.tight_layout()

In [ ]:
strain_id = 'Bacteroides-thetaiotaomicron-VPI-5482'

fig, ax = plt.subplots()

clust_id = cross_correlation.loc[strain_id].idxmax()

max_val = max(x[strain_id].max(), y[clust_id].max())

ax.scatter(x[strain_id], y[clust_id])
ax.plot([0, max_val], [0, max_val])
ax.set_xlabel(strain_id)
ax.set_ylabel(f'bin {clust_id}')
ax.annotate('{:0.01e}'.format(clust_meta.loc[clust_id].total_length), xy=(0.6, 0.3), xycoords='axes fraction')    
# ax.set_xscale('symlog', linthresh=1e-5)
# ax.set_yscale('symlog', linthresh=1e-5)
ax.set_aspect(1)
# fig.tight_layout()
cross_correlation.loc[strain_id].sort_values(ascending=False).head()

In [ ]:
focal_clust = 20

focal_clusts = [
    focal_clust,
    # 2383,
    338,
    332,
]

clust_palette = lib.plot.construct_ordered_palette(focal_clusts, cm='tab20')

focal_clust_segments = idxwhere(segement_x_clust == focal_clust)

d = final_results.loc[idxwhere(vertex_to_clust.isin(focal_clusts))].sort_values('length', ascending=False)#.head(20)

for clust in focal_clusts:
    color = clust_palette[clust]
    plt.scatter('length', 'total_depth', data=d.loc[idxwhere(vertex_to_clust == clust)], s=4, label=clust, color=color)
plt.xscale('log')
plt.yscale('log')
plt.legend()

print(d.length.sum())
d.join(vertex_to_clust).sort_values('num_segments', ascending=False)

In [ ]:
_col_colors = d.length.pipe(np.log10).pipe(lambda x: mpl.cm.viridis(x / x.max()))
# _col_colors = d.index.to_series().map(vertex_to_clust).map(clust_palette)

sns.clustermap(
    vertex_depth.loc[d.index].T.rename(sample_idx_to_id),
    col_colors=_col_colors, metric='cosine',
    norm=mpl.colors.SymLogNorm(linthresh=1)
)

In [ ]:
cg = sns.clustermap(
    vertex_depth.loc[idxwhere(d.length > 1000)].T.rename(sample_idx_to_id),
    metric='cosine',
    norm=mpl.colors.SymLogNorm(linthresh=0.01),
    figsize=(6, 2.5),
    dendrogram_ratio=(0, 0.2),
    xticklabels=0, yticklabels=0,
)

cg.ax_cbar.set_visible(False)

In [ ]:
clust_neighborhood_segment_graph = sz.io.load_graph(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.unassembled-c{focal_clust}-r100.notips-1-unpressed.sz')
sz.results.extract_vertex_data(clust_neighborhood_segment_graph).sort_values('length', ascending=False)#.head(20)

In [ ]:
_graph = clust_neighborhood_segment_graph


sz.draw.update_xypositions(_graph)
print("Done with layout.")

# Color by depth
vertex_color = _graph.new_vertex_property('float', vals=np.log(sz.results.total_depth_property(_graph).a + 1))

# Highlight vertices in the cluster itself.
_vertices = list(sz.results.iter_find_vertices_with_any_segment(_graph, focal_clust_segments))
print(len(_vertices))
in_path = _graph.new_vertex_property('bool', val=False)
in_path.a[_vertices] = 1

outpath = f'fig/run-{run_number}/clust-{focal_clust}.pdf'
sz.draw.draw_graph(
    _graph,
    # vertex_text=_graph.vp['length'],
    vertex_text=_graph.vp['sequence'],
    vertex_halo=in_path,
    # vertex_font_size=5,
    vertex_fill_color=vertex_color,
    # edge_pen_width=10,
    output=outpath,
    vcmap=(mpl.cm.magma, 1),
    output_size=(1000, 1000),
)
print(outpath)

In [ ]:
segment_x_cluster = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.segment.tsv')

In [ ]:
%%time
radius = 10

focal_clusts = [
    20,
    # 2383,
    338,
    332,
]

# Select segments
_segment_list = list(segment_x_cluster[lambda x: x.cluster.isin(focal_clusts)].segment.drop_duplicates().values)

# Select vertices
_vertex_list = list(sz.results.iter_find_vertices_with_any_segment(unpressed_graph, _segment_list))
# Find neighborhood
_distance_from_focal = sz.topology.get_shortest_distance_to_any_vertex(unpressed_graph, _vertex_list, unpressed_graph.vp['length'])
_vertex_mask = unpressed_graph.new_vertex_property('bool', vals=_distance_from_focal.a <= radius)
_neighborhood_vertex_list = idxwhere(pd.DataFrame(unpressed_graph.get_vertices(vprops=[_vertex_mask]), columns=['vertex', 'mask']).set_index('vertex')['mask'] == 1)
_graph0 = gt.Graph(gt.GraphView(unpressed_graph, vfilt=_vertex_mask), prune=True)
print(_graph0)

# Layout graph
gt.seed_rng(1)
np.random.seed(2)
sz.draw.update_xypositions(_graph0, layout=gt.draw.random_layout)  # NOTE: This allows for reproducible layouts.
print("Done with Step 1.")
sz.draw.update_xypositions(
    _graph0,
    layout=gt.draw.sfdp_layout,
    # p=1.4,
    # K=1,
)
print("Done with layout!")

In [ ]:
# _palette = {332: 'fuchsia', 2383: 'silver', 20: 'lime', 338: 'cyan'}
_palette = {332: 'hotpink', 2383: 'silver', 20: 'limegreen', 338: 'dodgerblue'}

In [ ]:
# Decorate vertices
_results = sz.results.extract_vertex_data(_graph0)
_segment_to_vertex = _results.segments.str[0].reset_index().set_index('segments').vertex.rename_axis(index='segment')

clust_membership = (
    final_results[lambda x: x.clust.isin(focal_clusts)]
    .groupby('clust')
    .apply(lambda x: x.segments.explode().drop_duplicates(), include_groups=False)
    .rename('segment')
    .reset_index()
    .set_index(['clust', 'segment'])
    .assign(tally=1)
    .tally
    .unstack(fill_value=0)
    .T
    .rename(_segment_to_vertex)
    .reindex(_graph0.get_vertices(), fill_value=0)
    # .sort_index()
)

In [ ]:
# num_clusts = 3
# _clusts1 = focal_clusts[:num_clusts]

_clust_order = [
    332,
    20,
    # 2383,
    338,
]

_clusts1 = _clust_order
_graph1 = _graph0.copy()

# # Select vertices in bin.
# _clusts1 = _clust_order[:i + 1]
# _mask1 = sz.results.extract_vertex_data(_graph0).segments.str[0].isin(segment_x_cluster[lambda x: x.cluster.isin(_clusts1)].segment)
# print(_mask1.mean())
# _graph1 = gt.GraphView(_graph0, vfilt=_mask1)

# # Decorate vertices
# _clust_mem = clust_membership[_mask1][_clusts1]
# _clust_frac1 = _clust_mem.divide(_clust_mem.sum(axis=1), axis=0)
# _graph1.vp['clust_frac'] = _graph1.new_vertex_property('vector<float>', vals=_clust_frac1.sort_index().values)

# Decorate vertices
_color_subset = sz.results.extract_vertex_data(_graph0).segments.str[0].isin(segment_x_cluster[lambda x: x.cluster.isin(_clusts1)].segment)
_clust_mem = clust_membership[_color_subset][_clusts1]
_clust_frac1 = _clust_mem.divide(_clust_mem.sum(axis=1), axis=0).reindex(_graph1.get_vertices(), fill_value=0)
_clust_frac_vprop = _graph1.new_vertex_property('vector<float>', vals=_clust_frac1.sort_index().values)

outpath = f'fig/run-{run_number}/Veill-multibin.all.png'
sz.draw.draw_graph(
    _graph1, output=outpath,
    vertex_text='',
    # vertex_color='k',
    vertex_size=30,
    vertex_shape='pie',
    vertex_pie_fractions=_clust_frac_vprop,
    vertex_pie_colors=[_palette[c] for c in _clusts1],
    output_size=(2000, 2000),
)
print(outpath)

In [ ]:
# num_clusts = 3
# _clusts1 = focal_clusts[:num_clusts]

_clust_order = [
    332,
    20,
    # 2383,
    338,
]

for c in _clust_order:
    _clusts1 = [c]
    _graph1 = _graph0.copy()
    
    # # Select vertices in bin.
    # _clusts1 = _clust_order[:i + 1]
    # _mask1 = sz.results.extract_vertex_data(_graph0).segments.str[0].isin(segment_x_cluster[lambda x: x.cluster.isin(_clusts1)].segment)
    # print(_mask1.mean())
    # _graph1 = gt.GraphView(_graph0, vfilt=_mask1)

    # # Decorate vertices
    # _clust_mem = clust_membership[_mask1][_clusts1]
    # _clust_frac1 = _clust_mem.divide(_clust_mem.sum(axis=1), axis=0)
    # _graph1.vp['clust_frac'] = _graph1.new_vertex_property('vector<float>', vals=_clust_frac1.sort_index().values)
    
    # Decorate vertices
    _color_subset = sz.results.extract_vertex_data(_graph0).segments.str[0].isin(segment_x_cluster[lambda x: x.cluster.isin(_clusts1)].segment)
    _clust_mem = clust_membership[_color_subset][_clusts1]
    _clust_frac1 = _clust_mem.divide(_clust_mem.sum(axis=1), axis=0).reindex(_graph1.get_vertices(), fill_value=0)
    _clust_frac_vprop = _graph1.new_vertex_property('vector<float>', vals=_clust_frac1.sort_index().values)
    
    outpath = f'fig/run-{run_number}/Veill-multibin.{c}.png'
    sz.draw.draw_graph(
        _graph1, output=outpath,
        vertex_text='',
        # vertex_color='k',
        vertex_size=30,
        vertex_shape='pie',
        vertex_pie_fractions=_clust_frac_vprop,
        vertex_pie_colors=[_palette[c] for c in _clusts1],
        output_size=(2000, 2000),
    )
    print(outpath)

In [ ]:
shared_segments = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.smoothed-6.unzip-{deconv}.clust-{clust_params}.shared.tsv')#.set_index(['clusterA', 'clusterB']).num_shared_segments.unstack(fill_value=0).reindex(index=clust_meta.index, columns=clust_meta.index, fill_value=0)
shared_segments.sort_values('num_shared_segments', ascending=False)#[lambda x: x.clusterA.isin([212]) | x.clusterB.isin([212])].head(10)
# shared_segments.values[:,:] += shared_segments.values.T
# shared_segments.max().sort_values()

In [ ]:
shared_segments[lambda x: x.clusterA == 98].sort_values('num_shared_segments', ascending=False).head()

In [ ]:
clust_meta2 = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.clust-{clust_params}.meta.tsv', index_col='cluster')
clust_depth2 = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.clust-{clust_params}.depth.tsv').rename(columns={'Unnamed: 0': 'cluster'}).set_index('cluster').T.rename(int)

clust_meta2.sort_values('total_length', ascending=False).head(10)

In [ ]:
vertex_depth2 = sz.results.depth_table(notips_unsmoothed_graph, notips_unsmoothed_graph.get_vertices()).T

In [ ]:
# Normalize by rpoB depth
marker_genes2 = pd.read_table(f'data/group/{group}/r.proc.kmtricks-{kmtricks}.ggcat-{graph_type}.notips-2.cds.tran.hmmer-{marker_model}-ga.tsv', names=['orf', 'gene_name', 'bitscore']).assign(vertex=lambda x: x.orf.str.split('_').str[0].astype(int))
_marker_gene_vertex_list = list(set(marker_genes2.vertex))
marker_depth2 = vertex_depth2.loc[_marker_gene_vertex_list].T

In [ ]:
clust_rabund2 = clust_depth2.divide(marker_depth2.sum(1), axis=0)

In [ ]:
x = kraken_rabund
y = clust_rabund2[idxwhere(clust_meta2.total_length > 2_000_000)].rename(sample_idx_to_id)

x, y = x.align(y, join='inner', axis='index')

cross_correlation2 = pd.DataFrame(1 - sp.spatial.distance.cdist(x.T, y.T, metric='cosine'), index=x.columns, columns=y.columns)

sns.clustermap(cross_correlation2)

In [ ]:
x = kraken_rabund
y = clust_rabund2[idxwhere(clust_meta2.total_length > 2_000_000)].rename(sample_idx_to_id)

x, y = x.align(y, join='inner', axis='index')

_strain_id_list = kraken_rabund.columns
fig, axs = lib.plot.subplots_grid(ncols=5, naxes=len(_strain_id_list))
for strain_id, ax in zip(_strain_id_list, axs.flatten()):
    clust_id = cross_correlation2.loc[strain_id].idxmax()

    max_val = max(x[strain_id].max(), y[clust_id].max())

    ax.scatter(x[strain_id], y[clust_id])
    ax.plot([0, max_val], [0, max_val])
    ax.set_xlabel(strain_id)
    ax.set_ylabel(f'bin {clust_id}')
    ax.set_aspect(1)
    ax.annotate('{:0.01e}'.format(clust_meta2.loc[clust_id].total_length), xy=(0.6, 0.3), xycoords='axes fraction')
    # cross_correlation.loc[strain_id].sort_values(ascending=False).head()
    
fig.tight_layout()

In [ ]:
vertex_to_clust.loc[marker_gene_vertex_list].value_counts().head(20)